# Combined Optimization Data Analysis

This notebook analyzes the combined single-step and two-step optimization databases and summarizes optimizer performance on:

- two-qubit gate reduction
- depth reduction
- runtime

All summaries are restricted to circuits with successful results for all four individual optimizers and all sixteen two-step chains.

It uses only the four active optimizers:

- `QISKIT`
- `TKET`
- `Wisq Rules`
- `Wisq BQSKit`


In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

# Paper-style formatting constants.
FONT_FAMILY = "DejaVu Sans"
TITLE_SIZE = 30
SUBTITLE_SIZE = 50
AXIS_LABEL_SIZE = 25
TICK_LABEL_SIZE = 20
LEGEND_SIZE = 10
ANNOTATION_SIZE = 9
TABLE_FONT_SIZE = 11
Y_TICK_LABEL_SIZE = 20
SAVE_FIGURES = True
FIGURE_DPI = 300

SINGLE_BAR_FIGSIZE = (19, 5.6)
CHAIN_BAR_FIGSIZE = (24, 8.5)
SINGLE_SCATTER_FIGSIZE = (16, 6.2)
CHAIN_SCATTER_FIGSIZE = (18, 7.2)

COLOR_2Q = "#355F78"
COLOR_DEPTH = "#4F7A57"
COLOR_RUNTIME = "#A55D4D"
PAPER_GRID_COLOR = "#CDD5D9"
PAPER_TEXT_COLOR = "#24333D"
PAPER_EDGE_COLOR = "#657681"
PAPER_FIGURE_COLOR = "#FFFFFF"
PAPER_AXES_COLOR = "#F7F4EA"

OPTIMIZER_NAME_MAP = {
    "qiskit_standard": "QISKIT",
    "tket": "TKET",
    "wisq_rules": "WISQ-R",
    "wisq_bqskit": "WISQ-BQ",
}
OPTIMIZERS = list(OPTIMIZER_NAME_MAP.keys())

sns.set_theme(
    style="whitegrid",
    context="paper",
    rc={
        "font.family": FONT_FAMILY,
        "axes.titlesize": TITLE_SIZE,
        "axes.labelsize": AXIS_LABEL_SIZE,
        "xtick.labelsize": TICK_LABEL_SIZE,
        "ytick.labelsize": TICK_LABEL_SIZE,
        "legend.fontsize": LEGEND_SIZE,
        "figure.facecolor": PAPER_FIGURE_COLOR,
        "axes.facecolor": PAPER_AXES_COLOR,
        "axes.edgecolor": PAPER_EDGE_COLOR,
        "axes.labelcolor": PAPER_TEXT_COLOR,
        "text.color": PAPER_TEXT_COLOR,
        "xtick.color": PAPER_TEXT_COLOR,
        "ytick.color": PAPER_TEXT_COLOR,
        "grid.color": PAPER_GRID_COLOR,
        "grid.linewidth": 0.8,
    },
)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "dbs").exists():
    for candidate in Path.cwd().resolve().parents:
        if (candidate / "dbs").exists():
            REPO_ROOT = candidate
            break

SINGLE_DB = REPO_ROOT / "dbs" / "combined_fixed" / "data_single_opt" / "combined_single_opt.db"
SECOND_DB = REPO_ROOT / "dbs" / "combined_fixed" / "data_second_opt" / "combined_second_opt.db"
FIGURE_OUTPUT_DIR = REPO_ROOT / "data" / "analyze_data_figures"

print(f"repo root: {REPO_ROOT}")
print(f"single db: {SINGLE_DB}")
print(f"second db: {SECOND_DB}")
print(f"figure output dir: {FIGURE_OUTPUT_DIR}")

In [ ]:
def pretty_optimizer_name(name: str) -> str:
    return OPTIMIZER_NAME_MAP.get(str(name), str(name))


def pretty_chain_name(chain_name: str) -> str:
    parts = [pretty_optimizer_name(part) for part in str(chain_name).split("__") if part]
    return " --> ".join(parts)


def apply_paper_axes(ax, *, x_grid: bool = True, y_grid: bool = True) -> None:
    ax.set_facecolor(PAPER_AXES_COLOR)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(PAPER_EDGE_COLOR)
    ax.spines["bottom"].set_color(PAPER_EDGE_COLOR)
    ax.grid(axis="x", visible=x_grid, alpha=0.35)
    ax.grid(axis="y", visible=y_grid, alpha=0.50)
    ax.tick_params(axis="both", labelsize=TICK_LABEL_SIZE)


def finish_figure(fig, filename: str | None = None) -> None:
    fig.tight_layout(pad=0.9)
    if SAVE_FIGURES and filename:
        FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        output_path = FIGURE_OUTPUT_DIR / filename
        fig.savefig(output_path, dpi=FIGURE_DPI, bbox_inches="tight")
        print(f"saved figure: {output_path}")
    plt.show()
    plt.close(fig)


def display_pretty_table(frame: pd.DataFrame) -> None:
    display(frame.style.set_properties(**{"font-size": f"{TABLE_FONT_SIZE}pt"}))

In [ ]:
single_query = """
SELECT
    c.id AS circuit_id,
    c.name AS original_circuit,
    c.category AS category,
    c.source AS source,
    o.name AS optimizer,
    r.success,
    r.error_message,
    r.artifact_path,
    r.duration_seconds,
    r.input_depth,
    r.input_two_qubit_gates,
    r.input_two_qubit_depth,
    r.input_total_gates,
    r.output_depth,
    r.output_two_qubit_gates,
    r.output_two_qubit_depth,
    r.output_total_gates
FROM optimization_runs r
JOIN circuits c ON c.id = r.circuit_id
JOIN optimizers o ON o.id = r.optimizer_id
"""

second_query = """
SELECT
    c.id AS parent_circuit_id,
    c.name AS parent_circuit_name,
    c.original_circuit AS original_circuit,
    c.category AS category,
    c.source AS source,
    o.name AS second_optimizer,
    r.success,
    r.error_message,
    r.artifact_path,
    r.duration_seconds AS second_step_runtime_seconds,
    r.input_depth AS second_step_input_depth,
    r.input_two_qubit_gates AS second_step_input_two_qubit_gates,
    r.input_two_qubit_depth AS second_step_input_two_qubit_depth,
    r.input_total_gates AS second_step_input_total_gates,
    r.output_depth AS final_depth,
    r.output_two_qubit_gates AS final_two_qubit_gates,
    r.output_two_qubit_depth AS final_two_qubit_depth,
    r.output_total_gates AS final_total_gates
FROM optimization_runs r
JOIN circuits c ON c.id = r.circuit_id
JOIN optimizers o ON o.id = r.optimizer_id
"""

with sqlite3.connect(SINGLE_DB) as single_conn:
    single_df = pd.read_sql_query(single_query, single_conn)

with sqlite3.connect(SECOND_DB) as second_conn:
    second_df = pd.read_sql_query(second_query, second_conn)

single_df = single_df[single_df["optimizer"].isin(OPTIMIZERS)].copy()
single_df["success"] = single_df["success"].astype(int)
single_df["depth_reduction"] = single_df["input_depth"] - single_df["output_depth"]
single_df["depth_reduction_pct"] = 100 * single_df["depth_reduction"] / single_df["input_depth"]
single_df["twoq_reduction"] = single_df["input_two_qubit_gates"] - single_df["output_two_qubit_gates"]
single_df["twoq_reduction_pct"] = 100 * single_df["twoq_reduction"] / single_df["input_two_qubit_gates"]
single_df["optimizer_pretty"] = single_df["optimizer"].map(pretty_optimizer_name)

second_df["success"] = second_df["success"].astype(int)
second_df = second_df[second_df["second_optimizer"].isin(OPTIMIZERS)].copy()
second_df["first_optimizer"] = second_df["parent_circuit_name"].str.rsplit("__", n=1).str[-1]
second_df = second_df[second_df["first_optimizer"].isin(OPTIMIZERS)].copy()
second_df["chain_name"] = second_df["first_optimizer"] + "__" + second_df["second_optimizer"]

parent_cols = [
    "original_circuit",
    "optimizer",
    "success",
    "duration_seconds",
    "input_depth",
    "input_two_qubit_gates",
    "input_two_qubit_depth",
    "input_total_gates",
    "output_depth",
    "output_two_qubit_gates",
    "output_two_qubit_depth",
    "output_total_gates",
]

parent_single = single_df[parent_cols].rename(
    columns={
        "optimizer": "first_optimizer",
        "success": "first_step_success",
        "duration_seconds": "first_step_runtime_seconds",
        "input_depth": "initial_depth",
        "input_two_qubit_gates": "initial_two_qubit_gates",
        "input_two_qubit_depth": "initial_two_qubit_depth",
        "input_total_gates": "initial_total_gates",
        "output_depth": "first_step_output_depth",
        "output_two_qubit_gates": "first_step_output_two_qubit_gates",
        "output_two_qubit_depth": "first_step_output_two_qubit_depth",
        "output_total_gates": "first_step_output_total_gates",
    }
)

chain_df = second_df.merge(
    parent_single,
    on=["original_circuit", "first_optimizer"],
    how="left",
    validate="many_to_one",
)

chain_df["total_runtime_seconds"] = (
    chain_df["first_step_runtime_seconds"] + chain_df["second_step_runtime_seconds"]
)
chain_df["depth_reduction"] = chain_df["initial_depth"] - chain_df["final_depth"]
chain_df["depth_reduction_pct"] = 100 * chain_df["depth_reduction"] / chain_df["initial_depth"]
chain_df["twoq_reduction"] = chain_df["initial_two_qubit_gates"] - chain_df["final_two_qubit_gates"]
chain_df["twoq_reduction_pct"] = 100 * chain_df["twoq_reduction"] / chain_df["initial_two_qubit_gates"]
chain_df["chain_success"] = ((chain_df["first_step_success"] == 1) & (chain_df["success"] == 1)).astype(int)
chain_df["first_optimizer_pretty"] = chain_df["first_optimizer"].map(pretty_optimizer_name)
chain_df["second_optimizer_pretty"] = chain_df["second_optimizer"].map(pretty_optimizer_name)
chain_df["chain_pretty"] = chain_df["chain_name"].map(pretty_chain_name)

single_success_df = single_df[single_df["success"] == 1].copy()
chain_success_df = chain_df[chain_df["chain_success"] == 1].copy()

# Restrict the entire notebook to circuits with no failed candidate runs.
expected_single_optimizers = set(OPTIMIZERS)
single_status = single_df.groupby("original_circuit").agg(
    single_actions=("optimizer", "nunique"),
    single_successes=("success", "sum"),
)
single_complete_circuits = set(single_status.index[
    (single_status["single_actions"] == len(expected_single_optimizers))
    & (single_status["single_successes"] == len(expected_single_optimizers))
])

expected_chain_names = {f"{first}__{second}" for first in OPTIMIZERS for second in OPTIMIZERS}
chain_status = chain_df.groupby("original_circuit").agg(
    chain_actions=("chain_name", "nunique"),
    chain_successes=("chain_success", "sum"),
)
chain_complete_circuits = set(chain_status.index[
    (chain_status["chain_actions"] == len(expected_chain_names))
    & (chain_status["chain_successes"] == len(expected_chain_names))
])

complete_circuits = sorted(single_complete_circuits & chain_complete_circuits)
single_df = single_df[single_df["original_circuit"].isin(complete_circuits)].copy()
chain_df = chain_df[chain_df["original_circuit"].isin(complete_circuits)].copy()
single_success_df = single_df[single_df["success"] == 1].copy()
chain_success_df = chain_df[chain_df["chain_success"] == 1].copy()

print(f"single rows: {len(single_df):,}")
print(f"single successful rows: {len(single_success_df):,}")
print(f"double rows: {len(chain_df):,}")
print(f"double successful rows: {len(chain_success_df):,}")
print(f"complete circuits retained: {len(complete_circuits):,}")

In [ ]:
summary_overview = pd.DataFrame(
    [
        {
            "dataset": "single optimizer runs",
            "rows": len(single_df),
            "successful_rows": int(single_df["success"].sum()),
            "unique_original_circuits": single_df["original_circuit"].nunique(),
            "unique_actions": single_df["optimizer"].nunique(),
        },
        {
            "dataset": "two-step chains",
            "rows": len(chain_df),
            "successful_rows": int(chain_df["chain_success"].sum()),
            "unique_original_circuits": chain_df["original_circuit"].nunique(),
            "unique_actions": chain_df["chain_name"].nunique(),
        },
    ]
)
display_pretty_table(summary_overview)

## Complete-corpus individual-optimizer reliability

This table combines individual-optimizer runs with recorded second-step chain runs. Each failure is attributed only to the optimizer that failed, whether it was the initial or secondary optimization. A run at or above the 900-second cap or with timeout wording in its error message is counted as a timeout regardless of its stored success flag; all other non-successful runs are counted as other failures.

In [ ]:
TIMEOUT_SECONDS = 900.0

with sqlite3.connect(SINGLE_DB) as single_conn:
    corpus_single_runs = pd.read_sql_query(
        """
        SELECT
            o.name AS optimizer,
            r.success,
            r.duration_seconds,
            r.error_message
        FROM optimization_runs r
        JOIN optimizers o ON o.id = r.optimizer_id
        WHERE o.name IN ({})
        """.format(",".join("?" for _ in OPTIMIZERS)),
        single_conn,
        params=OPTIMIZERS,
    )

corpus_single_runs["timeout"] = (
    (corpus_single_runs["duration_seconds"] >= TIMEOUT_SECONDS)
    | corpus_single_runs["error_message"].fillna("").str.contains("timeout", case=False, regex=False)
)
corpus_single_runs["valid_success"] = (
    (corpus_single_runs["success"].astype(int) == 1)
    & ~corpus_single_runs["timeout"]
)
corpus_single_runs["other_failure"] = (
    ~corpus_single_runs["valid_success"]
    & ~corpus_single_runs["timeout"]
)

individual_reliability = (
    corpus_single_runs.groupby("optimizer")
    .agg(
        failed_runs=("valid_success", lambda values: int((~values).sum())),
        success_rate=("valid_success", lambda values: 100 * values.mean()),
        timeouts=("timeout", "sum"),
        other_failures=("other_failure", "sum"),
    )
    .reindex(OPTIMIZERS)
    .reset_index()
    .assign(optimizer=lambda frame: frame["optimizer"].map(pretty_optimizer_name))
    .rename(
        columns={
            "optimizer": "Strategy",
            "failed_runs": "Failed runs",
            "success_rate": "Success rate (%)",
            "timeouts": "Timeouts",
            "other_failures": "Other failures",
        }
    )
)


with sqlite3.connect(SECOND_DB) as second_conn:
    corpus_chain_steps = pd.read_sql_query(
        """
        SELECT
            o.name AS optimizer,
            r.success,
            r.duration_seconds,
            r.error_message
        FROM optimization_runs r
        JOIN optimizers o ON o.id = r.optimizer_id
        WHERE o.name IN ({})
        """.format(",".join("?" for _ in OPTIMIZERS)),
        second_conn,
        params=OPTIMIZERS,
    )

corpus_chain_steps["timeout"] = (
    (corpus_chain_steps["duration_seconds"] >= TIMEOUT_SECONDS)
    | corpus_chain_steps["error_message"].fillna("").str.contains("timeout", case=False, regex=False)
)
corpus_chain_steps["valid_success"] = (
    (corpus_chain_steps["success"].astype(int) == 1)
    & ~corpus_chain_steps["timeout"]
)
corpus_chain_steps["other_failure"] = (
    ~corpus_chain_steps["valid_success"]
    & ~corpus_chain_steps["timeout"]
)

chain_step_reliability = (
    corpus_chain_steps.groupby("optimizer")
    .agg(
        attempted_runs=("valid_success", "size"),
        failed_runs=("valid_success", lambda values: int((~values).sum())),
        success_rate=("valid_success", lambda values: 100 * values.mean()),
        timeouts=("timeout", "sum"),
        other_failures=("other_failure", "sum"),
    )
    .reindex(OPTIMIZERS)
    .reset_index()
    .assign(optimizer=lambda frame: frame["optimizer"].map(pretty_optimizer_name))
    .rename(
        columns={
            "optimizer": "Strategy",
            "attempted_runs": "Attempted chain steps",
            "failed_runs": "Failed runs",
            "success_rate": "Success rate (%)",
            "timeouts": "Timeouts",
            "other_failures": "Other failures",
        }
    )
)
all_optimizer_runs = pd.concat(
    [corpus_single_runs, corpus_chain_steps],
    ignore_index=True,
)
optimizer_reliability = (
    all_optimizer_runs.groupby("optimizer")
    .agg(
        total_executions=("valid_success", "size"),
        failed_runs=("valid_success", lambda values: int((~values).sum())),
        success_rate=("valid_success", lambda values: 100 * values.mean()),
        timeouts=("timeout", "sum"),
        other_failures=("other_failure", "sum"),
    )
    .reindex(OPTIMIZERS)
    .reset_index()
    .assign(optimizer=lambda frame: frame["optimizer"].map(pretty_optimizer_name))
    .rename(
        columns={
            "optimizer": "Strategy",
            "total_executions": "Total optimizer executions",
            "failed_runs": "Failed runs",
            "success_rate": "Success rate (%)",
            "timeouts": "Timeouts",
            "other_failures": "Other failures",
        }
    )
)
display_pretty_table(optimizer_reliability)

## Single-optimizer summaries

In [ ]:
single_counts = (
    single_df.groupby(["optimizer", "optimizer_pretty"], as_index=False)
    .agg(runs=("optimizer", "size"), success_rate=("success", "mean"))
)

single_metrics = (
    single_success_df.groupby(["optimizer", "optimizer_pretty"], as_index=False)
    .agg(
        avg_twoq_reduction_pct=("twoq_reduction_pct", "mean"),
        median_twoq_reduction_pct=("twoq_reduction_pct", "median"),
        avg_depth_reduction_pct=("depth_reduction_pct", "mean"),
        median_depth_reduction_pct=("depth_reduction_pct", "median"),
        avg_runtime_seconds=("duration_seconds", "mean"),
        median_runtime_seconds=("duration_seconds", "median"),
    )
)

single_summary = single_counts.merge(
    single_metrics,
    on=["optimizer", "optimizer_pretty"],
    how="left",
)
single_summary["success_rate"] = 100 * single_summary["success_rate"]
single_summary = single_summary.sort_values("avg_twoq_reduction_pct", ascending=False).reset_index(drop=True)
display_pretty_table(single_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=SINGLE_BAR_FIGSIZE, facecolor=PAPER_FIGURE_COLOR)

sns.barplot(data=single_summary, x="optimizer_pretty", y="avg_twoq_reduction_pct", ax=axes[0], color=COLOR_2Q, edgecolor=PAPER_EDGE_COLOR)
apply_paper_axes(axes[0], x_grid=False, y_grid=True)
axes[0].set_title("Mean 2Q Reduction", fontsize=TITLE_SIZE)
axes[0].set_xlabel("")
axes[0].set_ylabel("Mean 2Q reduction (%)", fontsize=AXIS_LABEL_SIZE)
axes[0].tick_params(axis="x", rotation=20, labelsize=TICK_LABEL_SIZE)
axes[0].tick_params(axis="y", labelsize=Y_TICK_LABEL_SIZE)

sns.barplot(data=single_summary, x="optimizer_pretty", y="avg_depth_reduction_pct", ax=axes[1], color=COLOR_DEPTH, edgecolor=PAPER_EDGE_COLOR)
apply_paper_axes(axes[1], x_grid=False, y_grid=True)
axes[1].set_title("Mean Depth Reduction", fontsize=TITLE_SIZE)
axes[1].set_xlabel("")
axes[1].set_ylabel("Mean depth reduction (%)", fontsize=AXIS_LABEL_SIZE)
axes[1].tick_params(axis="x", rotation=20, labelsize=TICK_LABEL_SIZE)
axes[1].tick_params(axis="y", labelsize=Y_TICK_LABEL_SIZE)


sns.barplot(data=single_summary, x="optimizer_pretty", y="avg_runtime_seconds", ax=axes[2], color=COLOR_RUNTIME, edgecolor=PAPER_EDGE_COLOR)
apply_paper_axes(axes[2], x_grid=False, y_grid=True)
axes[2].set_title("Mean Runtime", fontsize=TITLE_SIZE)
axes[2].set_xlabel("")
axes[2].set_ylabel("Mean runtime (s)", fontsize=AXIS_LABEL_SIZE)
axes[2].tick_params(axis="x", rotation=20, labelsize=TICK_LABEL_SIZE)
axes[2].tick_params(axis="y", labelsize=Y_TICK_LABEL_SIZE)


finish_figure(fig, "single_optimizer_summary_bars.png")

## Unique circuits won by each individual optimizer

In [ ]:
# One winner is assigned per circuit and metric. Ties use the first row in the existing data order.
winner_specs = {
    "2Q reduction": ("twoq_reduction_pct", "idxmax"),
    "depth reduction": ("depth_reduction_pct", "idxmax"),
    "runtime": ("duration_seconds", "idxmin"),
}

winner_frames = []
for metric_name, (metric_column, selector) in winner_specs.items():
    valid_rows = single_success_df.dropna(subset=[metric_column])
    winner_indices = valid_rows.groupby("original_circuit")[metric_column].agg(selector).dropna().astype(int)
    winners = valid_rows.loc[winner_indices, ["original_circuit", "optimizer_pretty"]].copy()
    winners["metric"] = metric_name
    winner_frames.append(winners)

winner_counts = (
    pd.concat(winner_frames)
    .groupby(["metric", "optimizer_pretty"], as_index=False)
    .agg(unique_circuits=("original_circuit", "nunique"))
)
display_pretty_table(winner_counts.pivot(index="optimizer_pretty", columns="metric", values="unique_circuits").fillna(0).astype(int))

fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True, facecolor=PAPER_FIGURE_COLOR)
for ax, metric_name in zip(axes, winner_specs):
    plot_data = winner_counts[winner_counts["metric"] == metric_name].sort_values("unique_circuits", ascending=False)
    sns.barplot(data=plot_data, x="optimizer_pretty", y="unique_circuits", ax=ax, color=COLOR_2Q, edgecolor=PAPER_EDGE_COLOR)
    apply_paper_axes(ax, x_grid=False, y_grid=True)
    ax.set_title(metric_name, fontsize=TITLE_SIZE)
    ax.set_xlabel("")
    ax.set_ylabel("Unique circuits won" if ax is axes[0] else "", fontsize=AXIS_LABEL_SIZE)
    ax.tick_params(axis="x", rotation=20, labelsize=TICK_LABEL_SIZE)

finish_figure(fig, "individual_optimizer_circuit_wins.png")

## Total individual-optimizer circuit wins

This figure counts only 2Q-reduction wins for each circuit. Ties are counted as a win for every optimizer sharing the best value.

In [ ]:
tie_aware_winner_specs = {
    "2Q reduction": ("twoq_reduction_pct", "max"),
}

tie_aware_wins = []
for metric_name, (metric_column, direction) in tie_aware_winner_specs.items():
    valid_rows = single_success_df.dropna(subset=[metric_column])
    for circuit, circuit_rows in valid_rows.groupby("original_circuit"):
        best_value = getattr(circuit_rows[metric_column], direction)()
        winning_rows = circuit_rows[circuit_rows[metric_column] == best_value]
        tie_aware_wins.extend(
            {"original_circuit": circuit, "optimizer_pretty": optimizer, "metric": metric_name}
            for optimizer in winning_rows["optimizer_pretty"]
        )

total_wins = (
    pd.DataFrame(tie_aware_wins)
    .groupby("optimizer_pretty", as_index=False)
    .agg(total_circuit_wins=("original_circuit", "size"))
    .set_index("optimizer_pretty")
    .reindex([pretty_optimizer_name(name) for name in OPTIMIZERS], fill_value=0)
    .reset_index()
)
display_pretty_table(total_wins)

fig, ax = plt.subplots(figsize=(12, 6), facecolor=PAPER_FIGURE_COLOR)
sns.barplot(
    data=total_wins,
    x="optimizer_pretty",
    y="total_circuit_wins",
    width=0.28,
    color=COLOR_2Q,
    edgecolor=PAPER_EDGE_COLOR,
    ax=ax,
)
apply_paper_axes(ax, x_grid=False, y_grid=True)
ax.set_title("Total Individual-Optimizer 2Q Wins", fontsize=TITLE_SIZE)
ax.set_xlabel("", fontsize=AXIS_LABEL_SIZE)
ax.set_ylabel("Total 2Q wins", fontsize=AXIS_LABEL_SIZE)
ax.tick_params(axis="x", rotation=20, labelsize=TICK_LABEL_SIZE)
for bar, value in zip(ax.patches, total_wins["total_circuit_wins"]):
    ax.annotate(
        f"{int(value)}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center",
        va="bottom",
        fontsize=ANNOTATION_SIZE,
        xytext=(0, 4),
        textcoords="offset points",
    )
finish_figure(fig, "total_individual_optimizer_circuit_wins.png")

## Two-step chain summaries

In [ ]:
chain_counts = (
    chain_df.groupby(
        ["chain_name", "chain_pretty", "first_optimizer", "first_optimizer_pretty", "second_optimizer", "second_optimizer_pretty"],
        as_index=False,
    )
    .agg(runs=("chain_name", "size"), success_rate=("chain_success", "mean"))
)

chain_metrics = (
    chain_success_df.groupby(
        ["chain_name", "chain_pretty", "first_optimizer", "first_optimizer_pretty", "second_optimizer", "second_optimizer_pretty"],
        as_index=False,
    )
    .agg(
        avg_twoq_reduction_pct=("twoq_reduction_pct", "mean"),
        median_twoq_reduction_pct=("twoq_reduction_pct", "median"),
        avg_depth_reduction_pct=("depth_reduction_pct", "mean"),
        median_depth_reduction_pct=("depth_reduction_pct", "median"),
        avg_total_runtime_seconds=("total_runtime_seconds", "mean"),
        median_total_runtime_seconds=("total_runtime_seconds", "median"),
    )
)

chain_summary = chain_counts.merge(
    chain_metrics,
    on=["chain_name", "chain_pretty", "first_optimizer", "first_optimizer_pretty", "second_optimizer", "second_optimizer_pretty"],
    how="left",
)
chain_summary["success_rate"] = 100 * chain_summary["success_rate"]
chain_summary = chain_summary.sort_values("avg_twoq_reduction_pct", ascending=False).reset_index(drop=True)
display_pretty_table(chain_summary)

## Total two-optimizer-chain circuit wins

This figure counts only 2Q-reduction wins for each circuit. Ties are counted as a win for every chain sharing the best value. The table lists all 16 chains by average 2Q reduction.

In [ ]:
chain_average_table = (
    chain_summary[[
        "chain_pretty",
        "avg_twoq_reduction_pct",
        "avg_depth_reduction_pct",
        "avg_total_runtime_seconds",
    ]]
    .sort_values("avg_twoq_reduction_pct", ascending=False)
    .rename(columns={
        "chain_pretty": "chain",
        "avg_twoq_reduction_pct": "avg_2q_reduction_pct",
        "avg_total_runtime_seconds": "avg_runtime_seconds",
    })
    .reset_index(drop=True)
)
display_pretty_table(chain_average_table)

chain_tie_aware_specs = {
    "2Q reduction": ("twoq_reduction_pct", "max"),
}

chain_tie_aware_wins = []
for metric_name, (metric_column, direction) in chain_tie_aware_specs.items():
    valid_rows = chain_success_df.dropna(subset=[metric_column])
    for circuit, circuit_rows in valid_rows.groupby("original_circuit"):
        best_value = getattr(circuit_rows[metric_column], direction)()
        winning_rows = circuit_rows[circuit_rows[metric_column] == best_value]
        chain_tie_aware_wins.extend(
            {"original_circuit": circuit, "chain_name": chain}
            for chain in winning_rows["chain_name"]
        )

chain_order = [f"{first}__{second}" for first in OPTIMIZERS for second in OPTIMIZERS]
chain_total_wins = (
    pd.DataFrame(chain_tie_aware_wins)
    .groupby("chain_name", as_index=False)
    .agg(total_circuit_wins=("original_circuit", "size"))
    .set_index("chain_name")
    .reindex(chain_order, fill_value=0)
    .reset_index()
)
chain_total_wins["chain_pretty"] = chain_total_wins["chain_name"].map(pretty_chain_name)
display_pretty_table(chain_total_wins[["chain_pretty", "total_circuit_wins"]])

chain_total_wins_plot = chain_total_wins.sort_values("total_circuit_wins", ascending=False)
fig, ax = plt.subplots(figsize=(15, 8), facecolor=PAPER_FIGURE_COLOR)
sns.barplot(
    data=chain_total_wins_plot,
    x="chain_pretty",
    y="total_circuit_wins",
    width=0.28,
    color=COLOR_2Q,
    edgecolor=PAPER_EDGE_COLOR,
    ax=ax,
)
apply_paper_axes(ax, x_grid=False, y_grid=True)
ax.set_title("Total Two-Optimizer-Chain 2Q Wins", fontsize=TITLE_SIZE)
ax.set_xlabel("", fontsize=AXIS_LABEL_SIZE)
ax.set_ylabel("Total 2Q wins", fontsize=AXIS_LABEL_SIZE)
ax.tick_params(axis="x", labelsize=15)
plt.setp(
    ax.get_xticklabels(),
    rotation=65,
    ha="right",
    rotation_mode="anchor",
)
for bar, value in zip(ax.patches, chain_total_wins_plot["total_circuit_wins"]):
    ax.annotate(
        f"{int(value)}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center",
        va="bottom",
        fontsize=ANNOTATION_SIZE,
        xytext=(0, 4),
        textcoords="offset points",
    )
finish_figure(fig, "total_two_optimizer_chain_circuit_wins.png")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=CHAIN_BAR_FIGSIZE, facecolor=PAPER_FIGURE_COLOR)

twoq_order = chain_summary.sort_values("avg_twoq_reduction_pct", ascending=False)
depth_order = chain_summary.sort_values("avg_depth_reduction_pct", ascending=False)
runtime_order = chain_summary.sort_values("avg_total_runtime_seconds", ascending=True)

sns.barplot(data=twoq_order, y="chain_pretty", x="avg_twoq_reduction_pct", ax=axes[0], color=COLOR_2Q, edgecolor=PAPER_EDGE_COLOR)
apply_paper_axes(axes[0], x_grid=True, y_grid=False)
axes[0].set_title("Mean 2Q Reduction", fontsize=TITLE_SIZE)
axes[0].set_xlabel("Mean 2Q reduction (%)", fontsize=AXIS_LABEL_SIZE)
axes[0].set_ylabel("")

sns.barplot(data=depth_order, y="chain_pretty", x="avg_depth_reduction_pct", ax=axes[1], color=COLOR_DEPTH, edgecolor=PAPER_EDGE_COLOR)
apply_paper_axes(axes[1], x_grid=True, y_grid=False)
axes[1].set_title("Mean Depth Reduction", fontsize=TITLE_SIZE)
axes[1].set_xlabel("Mean depth reduction (%)", fontsize=AXIS_LABEL_SIZE)
axes[1].set_ylabel("")

sns.barplot(data=runtime_order, y="chain_pretty", x="avg_total_runtime_seconds", ax=axes[2], color=COLOR_RUNTIME, edgecolor=PAPER_EDGE_COLOR)
apply_paper_axes(axes[2], x_grid=True, y_grid=False)
axes[2].set_title("Mean Total Runtime", fontsize=TITLE_SIZE)
axes[2].set_xlabel("Mean total runtime (s)", fontsize=AXIS_LABEL_SIZE)
axes[2].set_ylabel("")

finish_figure(fig, "two_step_chain_summary_bars.png")

## Success-rate tables

In [ ]:
display_pretty_table(single_summary[["optimizer_pretty", "runs", "success_rate"]].sort_values("success_rate", ascending=False))
display_pretty_table(chain_summary[["chain_pretty", "runs", "success_rate"]].sort_values("success_rate", ascending=False))

## Tradeoff scatter plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=SINGLE_SCATTER_FIGSIZE, facecolor=PAPER_FIGURE_COLOR)

sns.scatterplot(
    data=single_summary,
    x="avg_runtime_seconds",
    y="avg_twoq_reduction_pct",
    hue="optimizer_pretty",
    palette=[COLOR_2Q, COLOR_DEPTH, COLOR_RUNTIME, "#7B6DAF"],
    s=120,
    ax=axes[0],
)
apply_paper_axes(axes[0], x_grid=True, y_grid=True)
for _, row in single_summary.iterrows():
    axes[0].text(
        row["avg_runtime_seconds"],
        row["avg_twoq_reduction_pct"],
        f"  {row['optimizer_pretty']}",
        fontsize=ANNOTATION_SIZE,
    )
axes[0].set_title("Single Optimizers: Runtime vs 2Q Reduction", fontsize=TITLE_SIZE)
axes[0].set_xlabel("Mean runtime (s)", fontsize=AXIS_LABEL_SIZE)
axes[0].set_ylabel("Mean 2Q reduction (%)", fontsize=AXIS_LABEL_SIZE)
axes[0].legend(title="Optimizer", fontsize=LEGEND_SIZE, title_fontsize=LEGEND_SIZE)

sns.scatterplot(
    data=single_summary,
    x="avg_runtime_seconds",
    y="avg_depth_reduction_pct",
    hue="optimizer_pretty",
    palette=[COLOR_2Q, COLOR_DEPTH, COLOR_RUNTIME, "#7B6DAF"],
    s=120,
    ax=axes[1],
)
apply_paper_axes(axes[1], x_grid=True, y_grid=True)
for _, row in single_summary.iterrows():
    axes[1].text(
        row["avg_runtime_seconds"],
        row["avg_depth_reduction_pct"],
        f"  {row['optimizer_pretty']}",
        fontsize=ANNOTATION_SIZE,
    )
axes[1].set_title("Single Optimizers: Runtime vs Depth Reduction", fontsize=TITLE_SIZE)
axes[1].set_xlabel("Mean runtime (s)", fontsize=AXIS_LABEL_SIZE)
axes[1].set_ylabel("Mean depth reduction (%)", fontsize=AXIS_LABEL_SIZE)
axes[1].legend(title="Optimizer", fontsize=LEGEND_SIZE, title_fontsize=LEGEND_SIZE)

finish_figure(fig, "single_optimizer_tradeoff_scatter.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=CHAIN_SCATTER_FIGSIZE, facecolor=PAPER_FIGURE_COLOR)

sns.scatterplot(
    data=chain_summary,
    x="avg_total_runtime_seconds",
    y="avg_twoq_reduction_pct",
    hue="first_optimizer_pretty",
    style="second_optimizer_pretty",
    s=110,
    ax=axes[0],
)
apply_paper_axes(axes[0], x_grid=True, y_grid=True)
for _, row in chain_summary.iterrows():
    axes[0].text(
        row["avg_total_runtime_seconds"],
        row["avg_twoq_reduction_pct"],
        f"  {row['chain_pretty']}",
        fontsize=ANNOTATION_SIZE,
    )
axes[0].set_title("Chains: Runtime vs 2Q Reduction", fontsize=TITLE_SIZE)
axes[0].set_xlabel("Mean total runtime (s)", fontsize=AXIS_LABEL_SIZE)
axes[0].set_ylabel("Mean 2Q reduction (%)", fontsize=AXIS_LABEL_SIZE)
axes[0].legend(title="Chain components", fontsize=LEGEND_SIZE, title_fontsize=LEGEND_SIZE)

sns.scatterplot(
    data=chain_summary,
    x="avg_total_runtime_seconds",
    y="avg_depth_reduction_pct",
    hue="first_optimizer_pretty",
    style="second_optimizer_pretty",
    s=110,
    ax=axes[1],
)
apply_paper_axes(axes[1], x_grid=True, y_grid=True)
for _, row in chain_summary.iterrows():
    axes[1].text(
        row["avg_total_runtime_seconds"],
        row["avg_depth_reduction_pct"],
        f"  {row['chain_pretty']}",
        fontsize=ANNOTATION_SIZE,
    )
axes[1].set_title("Chains: Runtime vs Depth Reduction", fontsize=TITLE_SIZE)
axes[1].set_xlabel("Mean total runtime (s)", fontsize=AXIS_LABEL_SIZE)
axes[1].set_ylabel("Mean depth reduction (%)", fontsize=AXIS_LABEL_SIZE)
axes[1].legend(title="Chain components", fontsize=LEGEND_SIZE, title_fontsize=LEGEND_SIZE)

finish_figure(fig, "two_step_chain_tradeoff_scatter.png")

## Best performers by metric

In [ ]:
print("Best single optimizer by mean 2Q reduction:")
display_pretty_table(single_summary.nlargest(1, "avg_twoq_reduction_pct"))

print("Best single optimizer by mean depth reduction:")
display_pretty_table(single_summary.nlargest(1, "avg_depth_reduction_pct"))

print("Fastest single optimizer by mean runtime:")
display_pretty_table(single_summary.nsmallest(1, "avg_runtime_seconds"))

print("Best chain by mean 2Q reduction:")
display_pretty_table(chain_summary.nlargest(1, "avg_twoq_reduction_pct"))

print("Best chain by mean depth reduction:")
display_pretty_table(chain_summary.nlargest(1, "avg_depth_reduction_pct"))

print("Fastest chain by mean total runtime:")
display_pretty_table(chain_summary.nsmallest(1, "avg_total_runtime_seconds"))

## Ordered-chain complementarity

This 100% stacked bar chart shows whether the second optimizer improves, leaves unchanged, or worsens the two-qubit gate count produced by the first optimizer.

In [ ]:
chain_outcomes = chain_df[[
    "chain_name",
    "chain_pretty",
    "first_step_output_two_qubit_gates",
    "final_two_qubit_gates",
]].copy()
chain_outcomes["second_step_twoq_change"] = (
    chain_outcomes["first_step_output_two_qubit_gates"]
    - chain_outcomes["final_two_qubit_gates"]
)
chain_outcomes["outcome"] = np.select(
    [
        chain_outcomes["second_step_twoq_change"] > 0,
        chain_outcomes["second_step_twoq_change"] == 0,
    ],
    ["Improved", "Unchanged"],
    default="Worsened",
)

chain_order = [f"{first}__{second}" for first in OPTIMIZERS for second in OPTIMIZERS]
chain_labels = {
    chain_name: pretty_chain_name(chain_name)
    for chain_name in chain_order
}
outcome_order = ["Improved", "Unchanged", "Worsened"]
outcome_colors = {"Improved": COLOR_DEPTH, "Unchanged": "#9A9A9A", "Worsened": COLOR_RUNTIME}
outcome_percentages = (
    pd.crosstab(
        chain_outcomes["chain_name"],
        chain_outcomes["outcome"],
        normalize="index",
    )
    .reindex(index=chain_order, columns=outcome_order, fill_value=0)
    .mul(100)
)

fig, ax = plt.subplots(figsize=(18, 9), facecolor=PAPER_FIGURE_COLOR)
bottom = pd.Series(0.0, index=outcome_percentages.index)
for outcome in outcome_order:
    values = outcome_percentages[outcome]
    ax.bar(
        outcome_percentages.index.map(chain_labels),
        values,
        bottom=bottom,
        label=outcome,
        color=outcome_colors[outcome],
        edgecolor=PAPER_FIGURE_COLOR,
        linewidth=0.5,
    )
    bottom = bottom + values

apply_paper_axes(ax, x_grid=False, y_grid=True)
ax.set_title("Effect of the Second Optimizer on Two-Qubit Gates", fontsize=TITLE_SIZE)
ax.set_xlabel("Ordered optimizer chain", fontsize=AXIS_LABEL_SIZE)
ax.set_ylabel("Circuits (%)", fontsize=AXIS_LABEL_SIZE)
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=65, labelsize=12)
ax.legend(title="Second-step outcome", fontsize=LEGEND_SIZE, title_fontsize=LEGEND_SIZE)
finish_figure(fig, "ordered_chain_second_optimizer_outcomes.png")

## Pairwise optimizer complementarity

Each cell shows the average total two-qubit gate reduction percentage for that ordered optimizer chain, measured from the original circuit to the final chained circuit.

In [ ]:
pairwise = chain_df[[
    "first_optimizer",
    "second_optimizer",
    "twoq_reduction_pct",
    "first_step_output_two_qubit_gates",
    "final_two_qubit_gates",
]].copy()
pairwise["additional_twoq_reduction"] = (
    pairwise["first_step_output_two_qubit_gates"]
    - pairwise["final_two_qubit_gates"]
)
pairwise["additional_twoq_reduction_pct"] = np.where(
    pairwise["first_step_output_two_qubit_gates"] > 0,
    100 * pairwise["additional_twoq_reduction"]
    / pairwise["first_step_output_two_qubit_gates"],
    0.0,
)

def pairwise_matrix(value_column):
    matrix = (
        pairwise.groupby(["first_optimizer", "second_optimizer"])[value_column]
        .mean()
        .unstack()
        .reindex(index=OPTIMIZERS, columns=OPTIMIZERS)
    )
    matrix.index = [pretty_optimizer_name(name) for name in matrix.index]
    matrix.columns = [pretty_optimizer_name(name) for name in matrix.columns]
    return matrix

total_reduction_heatmap = pairwise_matrix("twoq_reduction_pct")
additional_reduction_heatmap = pairwise_matrix("additional_twoq_reduction_pct")

fig, axes = plt.subplots(2, 1, figsize=(11, 15), facecolor=PAPER_FIGURE_COLOR)
heatmap_specs = [
    (total_reduction_heatmap, "Mean Total 2Q Reduction (%)", "Full Chain 2Q Reduction"),
    (additional_reduction_heatmap, "Mean Improvement (%)", "Second Optimizer Improvement Over First-Step"),
]
panel_labels = ["(a)", "(b)"]

for ax, label, (matrix, colorbar_label, title) in zip(
    axes, panel_labels, heatmap_specs
):
    sns.heatmap(
        matrix,
        annot=True,
        fmt=".2f",
        annot_kws={
            "size": 20,
            "weight": "bold",
        },
        cmap="Blues",
        linewidths=0.6,
        linecolor=PAPER_FIGURE_COLOR,
        cbar_kws={"label": colorbar_label},
        ax=ax,
    )

    ax.set_title(title, fontsize=TITLE_SIZE)
    ax.set_xlabel("Second optimizer", fontsize=AXIS_LABEL_SIZE)
    ax.set_ylabel("First optimizer", fontsize=AXIS_LABEL_SIZE)
    ax.tick_params(axis="both", labelsize=TICK_LABEL_SIZE)

    # Add subplot label in the upper-left corner.
    ax.text(
        -0.12,
        1.05,
        label,
        transform=ax.transAxes,
        fontsize=22,
        fontweight="bold",
        va="bottom",
        ha="left",
    )

finish_figure(fig, "pairwise_optimizer_complementarity_heatmaps.png")

## Fixed versus oracle performance summary

Fixed methods apply one optimizer or chain to every circuit. Oracle methods select the best available optimizer or chain separately for each circuit. This table compares only 2Q and depth reduction.

In [ ]:
def complete_circuit_frame(frame, action_column):
    # Keep only circuits with every candidate action available, so fixed
    # and oracle results use the same circuit population.
    expected_actions = frame[action_column].nunique()
    complete_circuits = (
        frame.groupby("original_circuit")[action_column]
        .nunique()
        .loc[lambda counts: counts == expected_actions]
        .index
    )
    return frame[frame["original_circuit"].isin(complete_circuits)].copy()

def fixed_metric_value(frame, action_column, metric_column):
    per_action = frame.groupby([action_column, "original_circuit"])[metric_column].mean()
    return per_action.groupby(level=0).mean().max()

def oracle_metric_value(frame, metric_column):
    return frame.groupby("original_circuit")[metric_column].max().mean()

single_table_frame = complete_circuit_frame(single_success_df, "optimizer")
chain_table_frame = complete_circuit_frame(chain_success_df, "chain_name")

summary_table = pd.DataFrame({
    "Method class": [
        "Best fixed individual optimizer",
        "Best fixed two-optimizer chain",
        "Oracle individual optimizer",
        "Oracle two-optimizer chain",
    ],
    "2Q reduction": [
        fixed_metric_value(single_table_frame, "optimizer", "twoq_reduction_pct"),
        fixed_metric_value(chain_table_frame, "chain_name", "twoq_reduction_pct"),
        oracle_metric_value(single_table_frame, "twoq_reduction_pct"),
        oracle_metric_value(chain_table_frame, "twoq_reduction_pct"),
    ],
    "Depth reduction": [
        fixed_metric_value(single_table_frame, "optimizer", "depth_reduction_pct"),
        fixed_metric_value(chain_table_frame, "chain_name", "depth_reduction_pct"),
        oracle_metric_value(single_table_frame, "depth_reduction_pct"),
        oracle_metric_value(chain_table_frame, "depth_reduction_pct"),
    ],
})

display(summary_table.style.format({"2Q reduction": "{:.2f}%", "Depth reduction": "{:.2f}%"}).set_properties(**{"font-size": f"{TABLE_FONT_SIZE}pt"}))

## Best method length counts

For each circuit, this compares the best successful one-optimizer result with the best successful two-optimizer chain. Counts are printed separately for 2Q and depth reduction. A tie means both method lengths share the best value.

In [ ]:
single_candidates = single_success_df[["original_circuit", "twoq_reduction_pct", "depth_reduction_pct"]].copy()
single_candidates["method_length"] = "1 optimizer"
chain_candidates = chain_success_df[["original_circuit", "twoq_reduction_pct", "depth_reduction_pct"]].copy()
chain_candidates["method_length"] = "2 optimizer chain"
method_candidates = pd.concat([single_candidates, chain_candidates], ignore_index=True)

for metric_name, metric_column in [("2Q reduction", "twoq_reduction_pct"), ("depth reduction", "depth_reduction_pct")]:
    valid_candidates = method_candidates.dropna(subset=[metric_column])
    winner_labels = []
    for circuit, circuit_rows in valid_candidates.groupby("original_circuit"):
        best_value = circuit_rows[metric_column].max()
        best_lengths = set(circuit_rows.loc[circuit_rows[metric_column] == best_value, "method_length"])
        if best_lengths == {"1 optimizer"}:
            label = "1 optimizer"
        elif best_lengths == {"2 optimizer chain"}:
            label = "2 optimizer chain"
        else:
            label = "tie"
        winner_labels.append({"original_circuit": circuit, "method_length": label})
    winners = pd.DataFrame(winner_labels)
    counts = winners["method_length"].value_counts().reindex(["1 optimizer", "2 optimizer chain", "tie"], fill_value=0)
    print(f"{metric_name}:")
    print(counts.to_string())
    print(f"Total circuits counted: {len(winners):,}\n")